# 05 NVIDIA Earth-2 / FourCastNet Availability Check

This notebook screens whether NVIDIA Earth-2 / FourCastNet outputs are practically usable for the weather forecasting and Polymarket trading project.

The aim is not to run a full NVIDIA Earth-2 model locally at this stage. The aim is to check:

1. whether there is a directly downloadable forecast-output route;
2. whether the model output contains 2m temperature or related surface-temperature variables;
3. whether inference requires GPU/container infrastructure;
4. whether this source can realistically enter the Polymarket probability pipeline before the dissertation deadline.

This complements the AIFS / ECMWF output check.

## 1. Feasibility criteria

For this project, a weather-model output source is useful if it can provide:

- forecast run time;
- forecast valid time;
- forecast lead time;
- 2m temperature or equivalent surface-temperature output;
- geographical grid or point-location extraction;
- historical or reproducible forecast outputs;
- access without training the model from scratch.

The ideal source is precomputed and downloadable. A source that requires local GPU inference may still be useful, but it is less immediately practical.

In [3]:
import os
import sys
import json
import shutil
import subprocess
from pathlib import Path
from datetime import datetime, timezone

import requests
import pandas as pd

RAW_DIR = Path("../data/raw/nvidia_earth2")
PROCESSED_DIR = Path("../data/processed/nvidia_earth2")

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

def check_url(url, timeout=20):
    try:
        response = requests.get(url, timeout=timeout)
        return {
            "url": url,
            "status_code": response.status_code,
            "ok": response.ok,
            "content_type": response.headers.get("content-type"),
            "n_chars": len(response.text),
            "error": None,
        }
    except Exception as e:
        return {
            "url": url,
            "status_code": None,
            "ok": False,
            "content_type": None,
            "n_chars": None,
            "error": str(e)[:1000],
        }

def command_check(command):
    try:
        result = subprocess.run(
            command,
            shell=True,
            capture_output=True,
            text=True,
            timeout=20,
        )
        return {
            "command": command,
            "returncode": result.returncode,
            "stdout": result.stdout[:1000],
            "stderr": result.stderr[:1000],
        }
    except Exception as e:
        return {
            "command": command,
            "returncode": None,
            "stdout": None,
            "stderr": str(e)[:1000],
        }

## 2. Check NVIDIA Earth-2 / FourCastNet documentation endpoints

This is a lightweight check of whether the main public documentation and model pages are reachable.

In [6]:
nvidia_urls = [
    "https://docs.nvidia.com/nim/earth-2/fourcastnet/latest/overview.html",
    "https://docs.nvidia.com/nim/earth-2/fourcastnet/latest/api-reference.html",
    "https://docs.nvidia.com/nim/earth-2/fourcastnet/latest/quickstart-guide.html",
    "https://catalog.ngc.nvidia.com/orgs/nim/teams/nvidia/containers/fourcastnet",
    "https://catalog.ngc.nvidia.com/orgs/nvidia/teams/earth-2/models/fourcastnet3",
    "https://nvidia.github.io/earth2studio/",
]

url_check_df = pd.DataFrame([check_url(url) for url in nvidia_urls])
url_check_df

,url,status_code,ok,content_type,n_chars,error
0,https://docs.nvidia.com/nim/earth-2/fourcastne...,200,True,text/html,21232,None
1,https://docs.nvidia.com/nim/earth-2/fourcastne...,200,True,text/html,19910,None
2,https://docs.nvidia.com/nim/earth-2/fourcastne...,200,True,text/html,50650,None
3,https://catalog.ngc.nvidia.com/orgs/nim/teams/...,200,True,text/html; charset=utf-8,133070,None
4,https://catalog.ngc.nvidia.com/orgs/nvidia/tea...,200,True,text/html; charset=utf-8,140944,None
5,https://nvidia.github.io/earth2studio/,200,True,text/html; charset=utf-8,59331,None


## 3. Local infrastructure check

The FourCastNet NIM route appears to require running a local inference service. This section checks whether the current machine has the basic infrastructure that would normally be needed for local NVIDIA model deployment.

This is a feasibility check only. Failure here does not mean the model is irrelevant; it means this route is less convenient than direct forecast downloads such as ECMWF Open Data.

In [11]:
infrastructure_checks = [
    "docker --version",
    "nvidia-smi",
    "python -c \"import torch; print('torch_available', torch.cuda.is_available())\"",
]

infra_df = pd.DataFrame([command_check(cmd) for cmd in infrastructure_checks])
infra_df

,command,returncode,stdout,stderr
0,docker --version,127,,/bin/sh: docker: command not found\n
1,nvidia-smi,127,,/bin/sh: nvidia-smi: command not found\n
2,"python -c ""import torch; print('torch_availabl...",1,,"Traceback (most recent call last):\n File ""<s..."


## 4. Python package availability check

This checks whether common NVIDIA / Earth-2 packages are already available in the current Python environment.

No heavy installation is performed here.

In [14]:
packages = [
    "earth2studio",
    "torch",
    "xarray",
    "netCDF4",
    "cfgrib",
]

package_rows = []

for package in packages:
    try:
        __import__(package)
        package_rows.append({
            "package": package,
            "available": True,
            "error": None,
        })
    except Exception as e:
        package_rows.append({
            "package": package,
            "available": False,
            "error": str(e)[:500],
        })

package_df = pd.DataFrame(package_rows)
package_df

,package,available,error
0,earth2studio,False,No module named 'earth2studio'
1,torch,False,No module named 'torch'
2,xarray,True,None
3,netCDF4,False,No module named 'netCDF4'
4,cfgrib,False,No module named 'cfgrib'


## 5. Access-route assessment

This table summarises whether each NVIDIA Earth-2 / FourCastNet route is immediately usable for this dissertation.

In [17]:
assessment = pd.DataFrame([
    {
        "route": "FourCastNet NIM local inference",
        "what_it_requires": "NVIDIA NIM container, input array, local inference endpoint, likely NVIDIA GPU infrastructure",
        "2m_temperature_relevance": "Relevant if output variables include the required surface-temperature field",
        "historical_forecast_availability": "Not directly solved; inference would need historical input states",
        "usable_without_running_model": "No",
        "difficulty": "High",
        "current_verdict": "Useful but not immediate; likely too heavy before the empirical pipeline is stable",
        "possible_project_role": "Possible extension or supervisor-guided route"
    },
    {
        "route": "FourCastNet3 model assets",
        "what_it_requires": "Model assets and inference workflow; likely GPU or specialised environment",
        "2m_temperature_relevance": "Relevant surface/global output if variables can be mapped",
        "historical_forecast_availability": "Not direct unless precomputed outputs are found",
        "usable_without_running_model": "Unclear / likely no",
        "difficulty": "Medium to high",
        "current_verdict": "Important model candidate, but less immediate than AIFS Open Data",
        "possible_project_role": "Literature/model anchor; possible extension if accessible inference route is confirmed"
    },
    {
        "route": "Earth2Studio examples",
        "what_it_requires": "Earth2Studio setup and compatible hardware/software environment",
        "2m_temperature_relevance": "Potentially relevant depending on model and recipe",
        "historical_forecast_availability": "Depends on input datasets and recipes",
        "usable_without_running_model": "No",
        "difficulty": "Medium to high",
        "current_verdict": "Useful for future experimentation, not the first empirical source",
        "possible_project_role": "Development path if supervisors want NVIDIA route"
    },
    {
        "route": "Precomputed NVIDIA Earth-2 outputs",
        "what_it_requires": "Public downloadable forecast archive or accessible API endpoint",
        "2m_temperature_relevance": "Would be highly relevant if available",
        "historical_forecast_availability": "Unknown",
        "usable_without_running_model": "Yes, if found",
        "difficulty": "Unknown",
        "current_verdict": "Best NVIDIA route if available, but not yet identified",
        "possible_project_role": "High-value data source if discovered"
    },
])

assessment

,route,what_it_requires,2m_temperature_relevance,historical_forecast_availability,usable_without_running_model,difficulty,current_verdict,possible_project_role
0,FourCastNet NIM local inference,"NVIDIA NIM container, input array, local infer...",Relevant if output variables include the requi...,Not directly solved; inference would need hist...,No,High,Useful but not immediate; likely too heavy bef...,Possible extension or supervisor-guided route
1,FourCastNet3 model assets,Model assets and inference workflow; likely GP...,Relevant surface/global output if variables ca...,Not direct unless precomputed outputs are found,Unclear / likely no,Medium to high,"Important model candidate, but less immediate ...",Literature/model anchor; possible extension if...
2,Earth2Studio examples,Earth2Studio setup and compatible hardware/sof...,Potentially relevant depending on model and re...,Depends on input datasets and recipes,No,Medium to high,"Useful for future experimentation, not the fir...",Development path if supervisors want NVIDIA route
3,Precomputed NVIDIA Earth-2 outputs,Public downloadable forecast archive or access...,Would be highly relevant if available,Unknown,"Yes, if found",Unknown,"Best NVIDIA route if available, but not yet id...",High-value data source if discovered


## 6. Comparison with AIFS / ECMWF route

The AIFS / ECMWF route has already shown direct forecast-field download feasibility through ECMWF Open Data.

The NVIDIA Earth-2 route appears highly relevant but less immediately convenient if it requires local model inference or container deployment. The most useful next step is to ask whether there are precomputed NVIDIA Earth-2 / FourCastNet outputs that can be accessed directly.

In [20]:
comparison = pd.DataFrame([
    {
        "source": "AIFS / ECMWF Open Data",
        "direct_forecast_file_download": "Yes, tested",
        "requires_local_model_inference": "No",
        "2m_temperature_tested": "Yes, GRIB files downloaded",
        "city_level_processing_status": "Needs GRIB processing and grid extraction",
        "current_project_priority": "Highest empirical AI-weather route"
    },
    {
        "source": "NVIDIA Earth-2 / FourCastNet",
        "direct_forecast_file_download": "Not yet identified",
        "requires_local_model_inference": "Likely, unless precomputed output is available",
        "2m_temperature_tested": "Not yet",
        "city_level_processing_status": "Not reached",
        "current_project_priority": "Important feasibility check and possible extension"
    },
    {
        "source": "Open-Meteo ECMWF API",
        "direct_forecast_file_download": "JSON point forecasts available",
        "requires_local_model_inference": "No",
        "2m_temperature_tested": "Yes, city-level forecasts retrieved",
        "city_level_processing_status": "Already easy",
        "current_project_priority": "Practical fallback / prototype source"
    },
])

comparison

,source,direct_forecast_file_download,requires_local_model_inference,2m_temperature_tested,city_level_processing_status,current_project_priority
0,AIFS / ECMWF Open Data,"Yes, tested",No,"Yes, GRIB files downloaded",Needs GRIB processing and grid extraction,Highest empirical AI-weather route
1,NVIDIA Earth-2 / FourCastNet,Not yet identified,"Likely, unless precomputed output is available",Not yet,Not reached,Important feasibility check and possible exten...
2,Open-Meteo ECMWF API,JSON point forecasts available,No,"Yes, city-level forecasts retrieved",Already easy,Practical fallback / prototype source


## 7. Save outputs

In [23]:
url_check_df.to_csv(PROCESSED_DIR / "nvidia_earth2_url_checks.csv", index=False)
infra_df.to_csv(PROCESSED_DIR / "nvidia_earth2_infrastructure_checks.csv", index=False)
package_df.to_csv(PROCESSED_DIR / "nvidia_earth2_package_checks.csv", index=False)
assessment.to_csv(PROCESSED_DIR / "nvidia_earth2_access_route_assessment.csv", index=False)
comparison.to_csv(PROCESSED_DIR / "nvidia_earth2_vs_aifs_comparison.csv", index=False)

print("Saved NVIDIA Earth-2 / FourCastNet availability-check outputs. These files are ignored by git.")

Saved NVIDIA Earth-2 / FourCastNet availability-check outputs. These files are ignored by git.


## Current findings

NVIDIA Earth-2 / FourCastNet is clearly relevant to the project, but the immediately usable data route is less clear than the AIFS / ECMWF route.

The main findings are:

- The NVIDIA FourCastNet NIM route appears to be an inference/deployment route rather than a simple forecast-download route.
- Local inference would likely require container/GPU infrastructure and suitable input arrays.
- FourCastNet3 and Earth2Studio are important possible development routes, but they are not as immediately usable as ECMWF Open Data for near-term empirical work.
- The most useful NVIDIA-specific route would be a precomputed forecast archive or API that provides 2m temperature forecasts by run time, valid time, lead time and location. This has not yet been identified in this notebook.
- AIFS / ECMWF remains the strongest current AI-weather empirical route because 2m-temperature forecast-field downloads have already been tested successfully.

Questions for discussion:

1. Does NVIDIA Earth-2 provide precomputed 2m-temperature forecast outputs that can be downloaded without running local inference?
2. Should the empirical pipeline prioritise AIFS / ECMWF now, while keeping NVIDIA Earth-2 / FourCastNet as a possible extension?
3. If NVIDIA Earth-2 is considered important, is departmental GPU or cloud/container access available?
4. Should FourCastNet be used mainly as a literature/model-comparison anchor unless direct outputs are available?